# 第 5 周练习 — RAG 知识库

## 练习目标（理念）

**RAG**（Retrieval-Augmented Generation，检索增强生成）不是把整库塞进提示词，而是：

1. 把文档存成向量（embedding）
2. 提问时**检索**最相关的少数文档
3. 只把这些片段当作 context 交给模型作答

本笔记本：在本地 **Chroma** 向量库里索引 Insurellm 知识库（默认本地 MiniLM 嵌入，无云端 embedding 费用），再基于检索结果回答问题。

## 和本课 Week 5 的关系

| 概念 | 本练习里你会看到 |
|------|------------------|
| 知识库路径发现 | `Path` 向上查找 `week5/knowledge-base` |
| 向量库索引 | `chromadb.Client().create_collection` + `collection.add` |
| 检索 top-k | `collection.query(..., n_results=k)` |
| 受限作答 | system 要求只用 context，没有就说不知道 |

## 怎么跑

1. 确保课程仓库里有 `week5/knowledge-base`（含 `.md` 文档）
2. `.env` 里配置好可被 `OpenAI()` 读取的密钥（用于最后一步生成答案）
3. 从上到下运行；可改 `rag_answer` 里的问题列表做实验


In [1]:
# ========== 导入 + 环境 + 定位知识库目录 ==========

# 从 pathlib 导入 Path：跨平台拼路径、向上遍历父目录
from pathlib import Path
# 导入 chromadb：本地向量数据库（默认自带嵌入器）
import chromadb
# 从 openai 导入 OpenAI：后面用 frontier 模型根据检索上下文生成答案
from openai import OpenAI
# 从 dotenv 导入 load_dotenv：把 .env 密钥读进环境变量
from dotenv import load_dotenv

# 加载 .env（override=True：覆盖已有同名环境变量）
load_dotenv(override=True)
# 创建默认 OpenAI 客户端（密钥来自环境变量 OPENAI_API_KEY 等）
frontier = OpenAI()

# 从当前工作目录及其父目录向上找 week5/knowledge-base（换 cwd 也能找到）
KB = next(p / "week5/knowledge-base" for p in [Path.cwd(), *Path.cwd().parents]
          if (p / "week5/knowledge-base").exists())
# 打印实际用到的知识库路径，便于核对
print("knowledge base:", KB)


knowledge base: C:\Users\Nicholas Dean\projects\llm_engineering\week5\knowledge-base


In [2]:
# ========== 读入全部 .md，写入 Chroma collection（本地嵌入 + 存储） ==========

# 三个平行列表：文档正文 / 唯一 id / 元数据（与 collection.add 参数一一对应）
docs, ids, metas = [], [], []
# 递归找出知识库下所有 Markdown，按路径排序保证顺序稳定
for f in sorted(KB.rglob("*.md")):
    docs.append(f.read_text(encoding="utf-8"))           # the document text
    ids.append(str(f.relative_to(KB)))                   # a unique id (its relative path)
    metas.append({"category": f.parent.name})            # company / products / employees / contracts

# 新建内存型 Client，并创建名为 insurellm 的 collection
collection = chromadb.Client().create_collection("insurellm")
# add：内部会做 embedding，再把向量与原文一并入库
collection.add(documents=docs, ids=ids, metadatas=metas)  # embeds + stores all docs
# 打印入库条数，确认索引成功
print(f"indexed {collection.count()} documents")


indexed 76 documents


In [3]:
# ========== RAG 问答：先检索 top-k，再把 context 交给模型 ==========

def rag_answer(question, k=4):
    # query_texts：用问题文本检索；n_results=k 取最相关的 k 篇
    hits = collection.query(query_texts=[question], n_results=k)   # retrieve top-k relevant docs
    # 把命中文档用分隔线拼成一大段 context
    context = "\n\n---\n\n".join(hits["documents"][0])             # stitch them into context
    # system 限制「只用 context」；user 里同时放 Context 与 Question（prompt 字符串勿改）
    messages = [
        {"role": "system", "content": "Answer using ONLY the context. If it's not there, say you don't know."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]
    # 调用 gpt-4o-mini 生成答案（单次非流式）
    answer = frontier.chat.completions.create(model="gpt-4o-mini", messages=messages).choices[0].message.content
    # 返回：答案文本 + 命中文档 id 列表（便于溯源）
    return answer, ids_from(hits)

def ids_from(hits):
    # Chroma query 结果里 ids[0] 对应第一条 query 的命中 id 列表
    return hits["ids"][0]

# 用两个示例问题跑通 RAG 闭环：打印答案与 sources
for q in ["Who is the CEO of Insurellm?", "What products does Insurellm offer?"]:
    ans, sources = rag_answer(q)
    print(f"Q: {q}\nA: {ans}\n   sources: {sources}\n")


Q: Who is the CEO of Insurellm?
A: The CEO of Insurellm is Avery Lancaster.
   sources: ['company\\about.md', 'company\\overview.md', 'employees\\Avery Lancaster.md', 'company\\careers.md']



Q: What products does Insurellm offer?
A: Insurellm offers 8 insurance software products across multiple insurance lines:

### Core Insurance Portals
- **Carllm** - Auto insurance platform for insurers
- **Homellm** - Home insurance platform for insurers
- **Lifellm** - Life insurance platform with AI-powered underwriting
- **Healthllm** - Comprehensive health insurance platform
- **Bizllm** - Commercial insurance platform for business coverage

### Marketplace & Infrastructure
- **Markellm** - Marketplace connecting consumers with insurance providers (original flagship product)
- **Claimllm** - AI-powered claims processing platform across all insurance lines
- **Rellm** - Enterprise platform for the reinsurance sector
   sources: ['company\\about.md', 'company\\overview.md', 'contracts\\Contract with EverGuard Insurance for Rellm - AI-Powered Enterprise Reinsurance Solution.md', 'contracts\\Contract with National Claims Network for Claimllm.md']

